# Проверка исправлений аудита v5

Экспериментальная версия, улучшение качества пока не подтверждено. Целевая точка — перед кабины (центр передней грани кубоида как приближение).

Включите GPU T4. Ячейка восстановит код и данные из существующей папки MyDrive/Peleng-Colab, выполнит полный Way1 и затем scene2, сохранит результаты и журналы в audit-benchmark-v5. Обучение не запускается. Код закреплён на коммите 7e09d6baa3d3683141729e3ac214282bb6b98ae6.

Пропуски нужно учитывать вместе с ошибками; scene2 не имеет эталона. Для повторной проверки только одной сцены измените список case на ['way1-generic'] или ['scene2-front'].


In [ ]:
# @title Проверка аудита v5: Way1 и scene2, перед кабины
from pathlib import Path
import subprocess,sys,signal,json,hashlib,torch
ROOT=Path('/content/drive/MyDrive/Peleng-Colab')
SOURCE=Path('/content/peleng'); CODE=Path('/content/peleng-audit-v5')
COMMIT='7e09d6baa3d3683141729e3ac214282bb6b98ae6'
assert torch.cuda.is_available(), 'Нужен GPU Colab'
if not SOURCE.exists(): subprocess.run(['git','clone','https://github.com/petrussia/3d-track-detection-and-tracking.git',str(SOURCE)],check=True)
subprocess.run(['git','-C',str(SOURCE),'fetch','origin','codex/audit-observation-contract'],check=True)
if not CODE.exists(): subprocess.run(['git','-C',str(SOURCE),'worktree','add','--detach',str(CODE),COMMIT],check=True)
assert subprocess.check_output(['git','-C',str(CODE),'rev-parse','HEAD'],text=True).strip()==COMMIT
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(CODE/'requirements-colab.txt')],check=True)
from google.colab import drive
drive.mount('/content/drive')
sys.path.insert(0,str(CODE))
from peleng.colab_support import unpack_bundles
items=[x for x in json.loads((ROOT/'bundle-manifest.json').read_text()) if x['name']=='organizer']
stage=Path('/content/organizer-bundle-only'); stage.mkdir(exist_ok=True)
(stage/'bundle-manifest.json').write_text(json.dumps(items))
for part in items[0]['parts']:
    link=stage/part
    if not link.exists(): link.symlink_to(ROOT/part)
unpack_bundles(stage,'/content/peleng-data')
unpack_bundles(ROOT/'scene2-input','/content/peleng-scene2')
OUT=Path('/content/audit-benchmark-v5'); BACKUP=ROOT/'audit-benchmark-v5'; BACKUP.mkdir(exist_ok=True)
(BACKUP/'pip-freeze.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
print('Новый YOLO-проход. Кандидаты от 0.20; геометрия отдельно проверяет пригодность. Целевая точка: перед кабины.',flush=True)
for case in ['way1-generic','scene2-front']:
    command=[sys.executable,'-u',str(CODE/'scripts/run_geometry_benchmark.py'),
     '--way1','/content/peleng-data/datasets/organizer','--scene2','/content/peleng-scene2/datasets/scene2',
     '--way1-weights',str(ROOT/'yolo11s.pt'),'--scene2-weights',str(ROOT/'scene2-adaptation-v1/detector/weights/last.pt'),
     '--baseline-code','/content/peleng-baseline-23decea','--outputs',str(OUT),'--backup',str(BACKUP),'--only',case]
    with (BACKUP/(case+'-progress.log')).open('w') as log:
        process=subprocess.Popen(command,cwd=CODE,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in process.stdout:
                print(line,end='',flush=True);log.write(line);log.flush()
            if process.wait(): raise RuntimeError('Ошибка прогона '+case)
        except KeyboardInterrupt:
            process.send_signal(signal.SIGINT);process.wait();raise
print('Оба полных прогона завершены. Результаты:',BACKUP,flush=True)
